# Alden — engagement-rank prototype

Goal: given a **text-only** description of a video idea (caption + auto-generated description), predict a single **0-100 'worth making' rank** — how well the idea will perform on engagement, platform-bias removed.

This notebook builds the feature pipeline and runs **KNN** first so we can review a real number. LightGBM / RandomForest / XGBoost cells are in the notebook but left unexecuted until we look at KNN's results.

Input: `data/processed/processed.csv` (12,105 short-video posts, 100% have `description_json`).

## 1. Load & sanity-check the data

10-input `data/processed/processed.csv` (12,105 → 11,306 rows after removing the `views <= 0` rows, which had no valid engagement rate). All remaining rows have `views > 0` and a `description_json`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Anchor to the repo root (nbconvert runs from notebooks/, so walk up).
repo = Path.cwd()
while not (repo / 'data' / 'processed' / 'processed.csv').exists() and repo != repo.parent:
    repo = repo.parent
DATA = repo / 'data' / 'processed' / 'processed.csv'
print('loading', DATA)

pd.set_option('display.width', 160)
df = pd.read_csv(DATA)
print('shape:', df.shape)
print(df[['platform','views','engagement']].head())
print('\nplatform counts:')
print(df['platform'].value_counts())
print('\nrows with views<=0 (should be 0 after cleaning):', int((df['views']<=0).sum()))
print('  of those, by platform:'); print(df.loc[df['views']<=0,'platform'].value_counts())

## 2. Flatten `description_json` → numeric features

The JSON has 10 fields. 7 are small vocab lists → binary multi-hot; `people`/`brands` are huge (2,439 / 1,227) → keep values seen in ≥10 posts, collapse the rest to `_other`; `event` (126) multi-hots with a min-frequency cap. The 3-row `play_by-play` hyphen quirk is normalized to `play_by_play`.

In [ ]:
import json

js = df['description_json'].apply(json.loads)

LIST_FIELDS = ['content_theme','format_access','tone','context','overall_team',
               'audio_format','event','people','brands']
PROSE_FIELD = 'play_by_play'

# Normalize the 3-row hyphen quirk
for i, p in js.items():
    if 'play_by-play' in p and 'play_by_play' not in p:
        p['play_by_play'] = p.pop('play_by-play')

def tokens(p, k):
    v = p.get(k, [])
    return v if isinstance(v, list) else []

# Build vocabulary for each list field
recode = []
VOCAB = {}
for k in LIST_FIELDS:
    vals = set()
    for p in js:
        vals.update(tokens(p, k))
    VOCAB[k] = sorted(vals)
    recode.append(f'{k}: {len(vals)} unique')
print('vocabulary sizes:'); [print(' ', r) for r in recode]

# Multi-hot feature names with min-frequency collapse for high-cardinality fields
MIN_FREQ = 10
COLLAPSE = {'people', 'brands'}   # very high cardinality -> top-values + '_other'
feature_cols = []
mark_records = []
val_counts = {k: {} for k in LIST_FIELDS}
for p in js:
    for k in LIST_FIELDS:
        for v in tokens(p, k):
            val_counts[k][v] = val_counts[k].get(v, 0) + 1

for k in LIST_FIELDS:
    if k in COLLAPSE:
        keep = {v for v, c in val_counts[k].items() if c >= MIN_FREQ}
        cols = sorted(keep) + ['_other']
    else:
        cols = list(VOCAB[k])
    feature_cols.append((k, cols))

from collections import defaultdict
rec = defaultdict(dict)
for idx, p in js.items():
    for k, cols in feature_cols:
        toks = set(tokens(p, k))
        for c in cols:
            if c == '_other':
                keep = {v for v in toks if v in val_counts[k] and val_counts[k][v] >= MIN_FREQ}
                notable = {v for v in toks if v not in keep}
                val = int(bool(notable))
            else:
                val = int(c in toks)
            rec[idx][f'{k}_{c}'] = val

feat_df = pd.DataFrame.from_dict(rec, orient='index')
feat_df.index = df.index
print('\nmulti-hot feature matrix:', feat_df.shape)
print('sample cols:', list(feat_df.columns[:8]))

## 3. Embed the two prose fields

`play_by_play` (verbal description of the video) and the caption `content` are each embedded into a 384-dim vector using `all-MiniLM-L6-v2` (same embedder as rob.ipynb). This captures *meaning*, which a keyword one-hot cannot.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
texts_playbyplay = js.apply(lambda p: p.get(PROSE_FIELD, '') or '')
texts_caption   = df['content'].fillna('').astype(str)

print(f'embedding {len(df)} play_by_play strings...')
emb_pbp  = embedder.encode(texts_playbyplay.tolist(), show_progress_bar=True, batch_size=256)
print(f'embedding {len(df)} caption strings...')
emb_cap  = embedder.encode(texts_caption.tolist(),   show_progress_bar=True, batch_size=256)
print('emb_pbp', emb_pbp.shape, ' emb_cap', emb_cap.shape)

EMB_COLS = [f'pbp_e{i}' for i in range(emb_pbp.shape[1])] + [f'cap_e{i}' for i in range(emb_cap.shape[1])]
emb_df = pd.DataFrame(np.hstack([emb_pbp, emb_cap]), columns=EMB_COLS, index=df.index)

## 4. Assemble features + compute the target

**Target:** engagement rate = `engagement / views` → standardized within platform (removes platform/audience-size bias) → rescaled to a **0-100 percentile across all rows**. One single target column → model-agnostic, all-platform score.

**Features:** description multi-hots + 2×384 embeddings + `page` one-hot + `duration_seconds`. We deliberately do **not** feed `platform` as a feature, so the model can't learn 'YouTube = high' and must learn content quality.

In [ ]:
# --- features ---
page_oh = pd.get_dummies(df['page'], prefix='page').astype(int)
num = df[['duration_seconds']].copy()
X = pd.concat([feat_df, emb_df, page_oh, num], axis=1)
print('full feature matrix:', X.shape, '| #NaN:', int(X.isna().sum().sum()))

# --- video_id grouping key (leakage-safe CV) ---
from urllib.parse import urlparse, parse_qs
def video_id(url):
    u = str(url); p = urlparse(u)
    if 'youtube' in p.netloc or 'youtu.be' in p.netloc:
        return (parse_qs(p.query).get('v') or [''])[0] or u.rstrip('/').split('/')[-1]
    segs = [s for s in u.split('?')[0].split('#')[0].rstrip('/').split('/') if s]
    return segs[-1]
df['video_id'] = df['url'].apply(video_id)

# --- TARGET (rob.ipynb metric): PREDICT RAW engagement AND views together ---
# Success = MAE in raw engagement units, beating a median baseline.
Y = df[['engagement', 'views']].reset_index(drop=True).to_numpy()

X_tr = X.reset_index(drop=True)
Y_tr = Y
groups = df['video_id'].values
print('rows:', len(df), '| target shape:', Y.shape)
print(df[['engagement','views']].describe())

## 5. 5-fold grouped CV — KNN

`GroupedKFold` by `video_id` so a video never appears in both train and test. KNN needs feature scaling, so we standardize on the KNN copy only (trees don't need scaling). Metrics: **Spearman correlation** (ranking skill) and **MAE** (closeness in target units), plus RMSE for reference.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error
from sklearn.base import clone
import numpy as np

def _fit_predict(model, Xtr, ytr, Xte):
    """Fit & predict for a possibly-multi-output target.

    KNN / RandomForest / XGBoost accept a 2-D y (engagement, views) natively, so we
    use that. LightGBM only accepts 1-D y, so for 2-D targets we clone the model and
    fit one per column, then stack — keeping the SAME raw-engagement MAE metric."""
    m = clone(model)
    try:
        m.fit(Xtr, ytr)                       # native multi-output
        return np.asarray(m.predict(Xte))
    except ValueError:
        cols = []
        for c in range(ytr.shape[1]):         # 1-D-only model (LightGBM)
            mc = clone(model)
            mc.fit(Xtr, ytr[:, c])
            cols.append(mc.predict(Xte))
        return np.column_stack(cols)

def run_cv_raw(model, Xm, Ym, grp, name, feature_scaler=None):
    """5-fold grouped CV; predicts (engagement, views); scores the SAME metric for
    every model — MAE in RAW engagement units vs a median baseline (rob.ipynb)."""
    gkf = GroupKFold(n_splits=5)
    mae_raw_all, base_all = [], []
    for tr_idx, te_idx in gkf.split(Xm, groups=grp):
        Xtr, Xte = Xm[tr_idx], Xm[te_idx]
        if feature_scaler is not None:        # KNN needs scaled features; trees don't
            fs = feature_scaler()
            Xtr = fs.fit_transform(Xtr); Xte = fs.transform(Xte)
        tsc = RobustScaler()
        ytr = tsc.fit_transform(Ym[tr_idx]); yte = tsc.transform(Ym[te_idx])
        pred = tsc.inverse_transform(_fit_predict(model, Xtr, ytr, Xte))
        yte_raw = Ym[te_idx]
        mae_raw_all.append(mean_absolute_error(yte_raw[:, 0], pred[:, 0]))
        base_all.append(np.mean(np.abs(yte_raw[:, 0] - np.median(Ym[:, 0]))))
    m, b = np.mean(mae_raw_all), np.mean(base_all)
    print(f'{name}: MAE(raw engagement)={m:,.0f} +/-{np.std(mae_raw_all):,.0f} | '
          f'median-baseline={b:,.0f} | beats baseline by {b-m:,.0f} ({(b-m)/b*100:.0f}%)')

# ---- KNN — SAME metric function as LightGBM / RandomForest.
from sklearn.neighbors import KNeighborsRegressor
X_arr = X_tr.values
run_cv_raw(KNeighborsRegressor(n_neighbors=20, weights='distance'),
           X_arr, Y_tr, groups, 'KNN', feature_scaler=RobustScaler)

## 6. (Not run yet) LightGBM, RandomForest, XGBoost\n\nSuccess metric now **mirrors rob.ipynb**: predict RAW engagement AND views, score by **MAE expressed back in raw engagement units**, and compare against a **median baseline**. `run_cv_raw` is defined in the first code cell below; then run each model in its own cell.\n- **LightGBM**: seconds.\n- **RandomForest**: 100 trees, a few minutes.\n- **XGBoost**: skipped (needs `brew install libomp`).\n

In [15]:
# Section 6 — LightGBM and RandomForest.
# `run_cv_raw` (defined in the KNN cell above) is reused here, so every model reports
# the identical metric: raw-engagement MAE vs the median baseline.
# Trees pass no feature_scaler (gradient boosting doesn't need scaled features).

In [ ]:
# LightGBM — fast, seconds. Predicts (engagement, views); metric = raw-engagement MAE.
from lightgbm import LGBMRegressor
lgb = LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1)
run_cv_raw(lgb, X_tr.values, Y_tr, groups, 'LightGBM')

In [ ]:
# RandomForest — 100 trees + n_jobs=-1; a few minutes. Same raw-target metric.
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
run_cv_raw(rf, X_tr.values, Y_tr, groups, 'RandomForest')

# XGBoost — uncomment once 'brew install libomp' has been run on this machine
# import xgboost as xgb
# xgbm = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
# run_cv_raw(xgbm, X_tr.values, Y_tr, groups, 'XGBoost')

## 7. 2×2 grid CLASSIFICATION (mirrors rob.ipynb cells 49–66)

Alternative success metric: **bin `views` and `engagement` into low/high** (`N_BINS=2`
→ a 2×2 grid), and **classify** an idea into those quadrants. Scored by **accuracy vs
the majority-class baseline** — including **joint accuracy** (how often BOTH bins are
predicted right), exactly like rob.ipynb.

`run_cv_cls` trains **two binary classifiers per fold** (one head for the views bin, one
for the engagement bin), reusing the same 5-fold grouped CV and the same models as the
regression section, so the two success metrics are comparable.

In [22]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score
from sklearn.base import clone
import numpy as np

# --- bin views & engagement into low/high (2x2 grid) exactly as rob.ipynb ---
N_BINS = 2

def _iqr_binned(series, n=N_BINS):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    edges = np.linspace(q1 - 1.5*iqr, q3 + 1.5*iqr, n + 1)
    return np.clip(np.digitize(series.to_numpy(), edges) - 1, 0, n-1).astype(int)

views_bin       = _iqr_binned(pd.Series(Y_tr[:, 1]))   # views
engagement_bin  = _iqr_binned(pd.Series(Y_tr[:, 0]))   # engagement
y_views_cls      = views_bin
y_eng_cls        = engagement_bin
print('views bin distribution:'); print(pd.Series(views_bin).value_counts().sort_index())
print('engagement bin distribution:'); print(pd.Series(engagement_bin).value_counts().sort_index())
print('grid labels (views x engagement):')
print(pd.Series([f'v{v}_e{e}' for v,e in zip(views_bin, engagement_bin)]).value_counts())

def run_cv_cls(model_factory, Xm, y_views, y_eng, grp, name, feature_scaler=None):
    """5-fold grouped CV; trains a model per head (views-bin, engagement-bin);
    reports views/engagement/joint accuracy vs the majority-class baseline."""
    gkf = GroupKFold(n_splits=5)
    acc_v, acc_e, acc_joint = [], [], []
    for tr, te in gkf.split(Xm, groups=grp):
        Xtr, Xte = Xm[tr], Xm[te]
        if feature_scaler is not None:
            fs = feature_scaler(); Xtr = fs.fit_transform(Xtr); Xte = fs.transform(Xte)
        m_v = clone(model_factory()); m_v.fit(Xtr, y_views[tr]);  pv = m_v.predict(Xte)
        m_e = clone(model_factory()); m_e.fit(Xtr, y_eng[tr]);    pe = m_e.predict(Xte)
        tv, te_ = y_views[te], y_eng[te]
        acc_v.append(accuracy_score(tv, pv))
        acc_e.append(accuracy_score(te_, pe))
        acc_joint.append(np.mean((pv == tv) & (pe == te_)))
    av, ae, aj = np.mean(acc_v), np.mean(acc_e), np.mean(acc_joint)
    # majority baselines (most common class in TRAIN)
    maj_v = pd.Series(y_views).mode().iloc[0];  maj_e = pd.Series(y_eng).mode().iloc[0]
    base_v = np.mean(y_views == maj_v);  base_e = np.mean(y_eng == maj_e)
    base_j = np.mean((y_views == maj_v) & (y_eng == maj_e))
    print(f'{name}: views_acc={av:.3f}(base {base_v:.3f}) | eng_acc={ae:.3f}(base {base_e:.3f}) | '
          f'joint_acc={aj:.3f}(base {base_j:.3f}) | joint improvement +{aj-base_j:.3f}')

views bin distribution:
0    7074
1    4232
Name: count, dtype: int64
engagement bin distribution:
0    7201
1    4105
Name: count, dtype: int64
grid labels (views x engagement):
v0_e0    6591
v1_e1    3622
v1_e0     610
v0_e1     483
Name: count, dtype: int64


In [23]:
# KNN on the 2x2 grid — same RobustScaler feature scaling as the regression KNN.
from sklearn.neighbors import KNeighborsClassifier
run_cv_cls(lambda: KNeighborsClassifier(n_neighbors=20, weights='distance', n_jobs=-1),
           X_tr.values, y_views_cls, y_eng_cls, groups,
           'KNN', feature_scaler=RobustScaler)

KNN: views_acc=0.782(base 0.626) | eng_acc=0.797(base 0.637) | joint_acc=0.715(base 0.583) | joint improvement +0.132


In [24]:
# LightGBM on the 2x2 grid.
from lightgbm import LGBMClassifier
run_cv_cls(lambda: LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1),
           X_tr.values, y_views_cls, y_eng_cls, groups, 'LightGBM')

LightGBM: views_acc=0.806(base 0.626) | eng_acc=0.820(base 0.637) | joint_acc=0.735(base 0.583) | joint improvement +0.152


In [25]:
# RandomForest on the 2x2 grid.
from sklearn.ensemble import RandomForestClassifier
run_cv_cls(lambda: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
           X_tr.values, y_views_cls, y_eng_cls, groups, 'RandomForest')

RandomForest: views_acc=0.786(base 0.626) | eng_acc=0.788(base 0.637) | joint_acc=0.704(base 0.583) | joint improvement +0.121
